# 03 · Action CNN (Step 4: single-frame baseline, TensorFlow / Keras)

**Owner:** Ben · **Input:** the clips from `03a_action_dataset.ipynb` · **Output:** a pretrained CNN (EfficientNetB0 by default) that labels a player's **middle frame** as spike, set, dig, block or other, with its test scores.

**Why a single-frame model first?** It trains quickly, so it shakes out bugs in the data and training before the slower video models in Step 5. Its test score is also the bar the temporal models must beat, which is the core comparison in your report section.

### How this follows the course textbook (*The Deep Learning Workshop*, Packt 2020)

| Section here | Technique | Textbook |
|---|---|---|
| 4.1 | `tf.keras`, reproducible seeds (`np.random.seed`, `tf.random.set_seed`) | Preface; Ch. 3, Ex. 3.04 |
| 4.3 | `ImageDataGenerator` + `.flow()` on NumPy arrays; data augmentation (flip, shift, zoom, rotation, shear, `fill_mode='nearest'`) | Ch. 3, *Data Generator*, *Data Augmentation*, Ex. 3.04 |
| 4.4 | Transfer learning from `keras.applications` (`weights='imagenet', include_top=False`), `base_model.trainable = False`, new `Dense` softmax head | Ch. 3, *Transfer Learning*, Activity 3.02 |
| 4.4 | `Dense` + ReLU, `Dropout`, softmax output | Ch. 2, *Overfitting and Dropout*; Ch. 3, *CNNs with TensorFlow and Keras* |
| 4.4 | `Adam(0.001)`, `loss='sparse_categorical_crossentropy'`, `metrics=['accuracy']` | Ch. 2–3 |
| 4.7 | Fine-tuning: freeze the first `frozen_layers` layers, retrain the rest | Ch. 3, *Fine-Tuning* |
| 4.6–4.9 | `save_weights` / `load_weights`, `model.save` | Ch. 3, *Saving and Restoring Models* |
| 4.9 | Confusion matrix on the test set | Ch. 3–4 |

**Changes and additions beyond the book**, each explained where it's used: **EfficientNetB0 instead of VGG16** (same API, more accurate; set `BACKBONE = "vgg16"` to run the book's model as a comparison); global average pooling instead of `Flatten`; each model's own `preprocess_input` instead of `rescale=1./255`; BatchNormalization kept frozen while fine-tuning; a lower learning rate for fine-tuning; `class_weight`; macro-F1 (the metric promised in the project proposal); a one-batch sanity check; and a per-epoch loop so training can resume after a Colab disconnect.

Runtime: **GPU required** (Colab T4 or Kaggle). Run top to bottom.

## 4.1 Setup

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import applications as apps
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

SEED = 0                                   # same seeding as the textbook's exercises
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:                             # let TensorFlow share the GPU with other libraries (e.g. YOLO) later
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__, "| GPUs:", gpus or "none")
if not gpus: print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then restart.")

### Paths and settings

On Colab the clips are copied once from Drive to local disk (faster to read). Weights and results are written **straight to Drive**, so a disconnect never loses them. All settings live in `HP`; change them here, not inside the code, so every run's settings are saved with its results.

In [ ]:
env = os.environ.get
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/volleyball/action")
    CLIPS_DIR = Path("/content/action_clips")
    if not (CLIPS_DIR / "labels.csv").exists():
        t0 = time.time(); shutil.copytree(DRIVE / "data" / "action_clips", CLIPS_DIR)
        print(f"Copied clips to local disk in {time.time() - t0:.0f} s")
    OUT = DRIVE
elif IN_KAGGLE:
    CLIPS_DIR = Path(env("CLIPS_DIR", "/kaggle/input/action-clips/action_clips"))  # <- your dataset's path
    OUT = Path("/kaggle/working")
else:
    CLIPS_DIR = Path(env("CLIPS_DIR", "./work/action_clips"))
    OUT = Path(env("OUT_DIR", "./action_out"))

BACKBONE = env("BACKBONE", "efficientnetb0")      # "efficientnetb0" (default), "vgg16" (the book's), or "resnet50"
RUN = env("RUN_NAME", f"baseline_{BACKBONE}")
CKPT_DIR, RESULTS_DIR = OUT / "checkpoints" / RUN, OUT / "results" / RUN
for d in (CKPT_DIR, RESULTS_DIR): d.mkdir(parents=True, exist_ok=True)

HP = dict(
    backbone=BACKBONE,
    img_size=112,                          # crop size from 03a; all three backbones accept inputs from 32 px up
    batch_size=int(env("BATCH", 32)),
    epochs_head=int(env("EPOCHS_HEAD", 5)),        # transfer-learning phase (book trains 5 epochs)
    epochs_finetune=int(env("EPOCHS_FT", 15)),     # fine-tuning phase (stops early if no progress)
    lr_head=0.001,                         # book's Adam learning rate
    lr_finetune=1e-4,                      # 10x lower when pretrained layers are unfrozen (see 4.7)
    dense_units=256, dropout=0.5,
    patience=4,                            # fine-tuning stops after this many epochs without a better val macro-F1
    pretrained=env("PRETRAINED", "1") == "1",
    sanity_epochs=int(env("SANITY_EPOCHS", 50)),
)
CLASSES = ["spike", "set", "dig", "block", "other"]
MID = 8                                    # index of the labelled frame inside each 16-frame clip
# Each backbone: its keras.applications class, its own input preprocessing, and the first layer that is
# retrained during fine-tuning (4.7). Everything before that layer stays frozen, as in the book's Fine-Tuning section.
BACKBONES = {
    "efficientnetb0": (apps.EfficientNetB0, apps.efficientnet.preprocess_input, "block7a"),
    "vgg16":          (apps.VGG16,          apps.vgg16.preprocess_input,        "block5_conv1"),
    "resnet50":       (apps.ResNet50,       apps.resnet50.preprocess_input,     "conv5_block1"),
}
BASE_CLASS, preprocess_input, FINETUNE_FROM = BACKBONES[BACKBONE]
json.dump(HP, open(RESULTS_DIR / "hparams.json", "w"), indent=2)
print("clips:", CLIPS_DIR, "\nresults:", RESULTS_DIR, "\n", HP)

## 4.2 Load the middle frames

The baseline only needs frame 8 (the labelled frame) of each clip, so we load just that: about 37 KB per clip, a few hundred MB for the whole set, which fits in memory. Like the textbook's CIFAR-10 exercise, the data is then a NumPy array of images plus an array of integer labels.

In [ ]:
labels = pd.read_csv(CLIPS_DIR / "labels.csv")
class_weights = json.load(open(CLIPS_DIR / "class_weights.json"))

def load_middle_frames(split):
    g = labels[labels.split == split].sort_values(["shard", "idx"])
    X, y, keep = [], [], []
    for shard, rows in g.groupby("shard"):
        z = np.load(CLIPS_DIR / f"{split}_{shard:03d}.npz")
        X.append(z["X"][rows.idx.values, MID]); y.append(z["y"][rows.idx.values])
        keep.append(rows)
    return np.concatenate(X), np.concatenate(y).astype(np.int64), pd.concat(keep).reset_index(drop=True)

data = {s: load_middle_frames(s) for s in ["train", "val", "test"]}
for s, (X, y, _) in data.items():
    print(f"{s:5s} X={X.shape} counts={dict(zip(CLASSES, np.bincount(y, minlength=5).tolist()))}")

**Check:** shapes are `(N, 112, 112, 3)` and the counts match the table at the end of `03a`.

## 4.3 Data generators and augmentation (Ch. 3)

As in Exercise 3.04, an `ImageDataGenerator` produces augmented batches from the arrays with `.flow()`. Augmentation helps the model generalise: different gyms, camera angles and player positions within the crop. The settings are milder than the book's examples because players must stay recognisable in a small 112-pixel crop:

| Augmentation | Setting | Why |
|---|---|---|
| Horizontal flip | on | A spike from the left looks like a spike from the right |
| Vertical flip | **off** | Players are never upside down; it would teach the model nonsense |
| Width / height shift | 0.1 | Our tracker's boxes won't be perfectly centred |
| Zoom | 0.15 | Players at different distances from the camera |
| Rotation | 10° | Slight camera tilt (the book's 40° is too much for people) |
| Shear | 0.1 | Mild perspective changes |
| `fill_mode='nearest'` | as in the book | Fills pixels exposed by shifts and rotations |

**One deliberate difference from the book:** instead of `rescale=1./255`, we pass the backbone's own `preprocess_input`. Pretrained filters work best on input prepared the way they were trained (VGG16 and ResNet50 subtract channel means; EfficientNet does its scaling inside the model, so its `preprocess_input` leaves pixels as they are). **Validation and test data are never augmented**, so they get a generator with only the preprocessing. (Exercise 3.04 builds its validation generator from the augmenting one; don't copy that detail.)

In [ ]:
AUG = dict(horizontal_flip=True, width_shift_range=0.1, height_shift_range=0.1,
           zoom_range=0.15, rotation_range=10, shear_range=0.1, fill_mode="nearest")
train_img_gen = ImageDataGenerator(preprocessing_function=preprocess_input, **AUG)
val_img_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

def flow(split, train):
    X, y, _ = data[split]
    gen = train_img_gen if train else val_img_gen
    return gen.flow(X.astype("float32"), y, batch_size=HP["batch_size"], shuffle=train, seed=SEED)

# Preview: the same augmentation without preprocessing, so the images display normally
show = ImageDataGenerator(**AUG).flow(data["train"][0][:8].astype("float32"), data["train"][1][:8], batch_size=8, shuffle=False)
xb, yb = next(show)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for ax, img, lab in zip(axes, xb, yb):
    ax.imshow(img.astype("uint8")); ax.set_title(CLASSES[lab]); ax.axis("off")
fig.suptitle("augmented training samples"); plt.show()

## 4.4 The model: transfer learning (Ch. 3)

The textbook's recipe, with a more modern pretrained network:

1. Load a CNN pretrained on ImageNet with `include_top=False`: keep its convolutional layers, drop its 1,000-class head. **Default: EfficientNetB0.** In the Keras model table it scores 77.1% top-1 on ImageNet against VGG16's 71.3%, with 5.3 M parameters instead of 138 M, and runs at about the same speed on a GPU. It's loaded and frozen exactly like the book's VGG16; set `BACKBONE = "vgg16"` in 4.1 to run the book's own model as a comparison row in your results.
2. **Freeze** it (`base_model.trainable = False`).
3. `pooling="avg"` (an option of every `keras.applications` model) averages each of the backbone's feature maps into one number, giving a vector of 1,280 numbers per image (512 for VGG16). The book uses `Flatten` here; with EfficientNet that would feed 20,480 numbers into the next layer, adding millions of weights for no benefit. We name this vector `features`: Step 5 reuses it.
4. Add the new head: `Dense(256, relu)` → `Dropout(0.5)` → `Dense(5, softmax)`, as in Activity 3.02 plus the dropout from Ch. 2.
5. Compile with `Adam(0.001)`, `sparse_categorical_crossentropy` and `accuracy`.

**BatchNormalization:** EfficientNet and ResNet contain BatchNormalization layers, which keep running averages of their inputs. We call the backbone with `training=False`, so those averages stay frozen even after fine-tuning unfreezes the layers around them. With small batches, letting them update would undo much of what the pretrained network learned. (VGG16 has no such layers.)

In [ ]:
def make_model(pretrained=HP["pretrained"]):
    size = HP["img_size"]
    base_model = BASE_CLASS(input_shape=(size, size, 3), weights="imagenet" if pretrained else None,
                            include_top=False, pooling="avg")
    base_model.trainable = False
    inputs = layers.Input(shape=(size, size, 3))
    feats = base_model(inputs, training=False)          # BatchNorm layers stay in inference mode
    feats = layers.Identity(name="features")(feats)
    x = layers.Dense(HP["dense_units"], activation="relu")(feats)
    x = layers.Dropout(HP["dropout"])(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name=f"{BACKBONE}_baseline"), base_model

def compile_model(model, lr):
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.Adam(lr), metrics=["accuracy"])

def set_finetune(base_model, on):
    # Ch. 3 Fine-Tuning: freeze the first layers, retrain the rest. BatchNorm layers always stay frozen.
    base_model.trainable = on
    if on:
        frozen = next(i for i, l in enumerate(base_model.layers) if l.name.startswith(FINETUNE_FROM))
        for i, layer in enumerate(base_model.layers):
            layer.trainable = i >= frozen and not isinstance(layer, layers.BatchNormalization)

model, base_model = make_model()
compile_model(model, HP["lr_head"])
model.summary()
_first = next(i for i, l in enumerate(base_model.layers) if l.name.startswith(FINETUNE_FROM))
print(f"Fine-tuning will retrain {BACKBONE} layers {_first}-{len(base_model.layers) - 1} of {len(base_model.layers)} "
      f"(from '{base_model.layers[_first].name}'), BatchNorm excluded")

## 4.5 Sanity check: can it memorise a few images?

*(Not in the book; a standard debugging step.)* Train a throwaway copy on just 32 training images, without augmentation, then measure its accuracy on those same images (with dropout off). With ImageNet features it should reach at least 0.9, usually 1.0. If it doesn't, something is broken (labels misaligned with images, preprocessing), and a full run would only waste GPU time.

In [ ]:
def sanity_check(n=32, epochs=HP["sanity_epochs"]):
    m, _ = make_model(); compile_model(m, HP["lr_head"])
    X, y, _ = data["train"]
    idx = np.random.RandomState(SEED).choice(len(y), size=min(n, len(y)), replace=False)
    Xs = preprocess_input(X[idx].astype("float32"))
    m.fit(Xs, y[idx], epochs=epochs, batch_size=n, verbose=0)
    acc = m.evaluate(Xs, y[idx], verbose=0)[1]        # measured with dropout off
    print(f"accuracy on the 32 images after {epochs} epochs: {acc:.2f}")
    print("OK: the model can learn from this data." if acc >= 0.9 else
          "WARNING: it could not memorise 32 images. Check that the ImageNet weights loaded (4.4 should not say "
          "'weights=None'), that BACKBONE matches its preprocessing, and that the 03a clip grid (3.4) looks right.")
    del m

sanity_check()

**Check:** the cell prints `OK`. If it prints a warning, stop and fix the data first.

## 4.6 Phase 1: train the new head (transfer learning)

Only the new layers learn; the backbone stays frozen. Instead of one `model.fit(epochs=N)` call, we call `fit` for **one epoch at a time** in a loop. After each epoch we:

1. predict the **validation** set and compute **macro-F1** (the F1 of each class, averaged equally, so a model that ignores blocks can't hide behind good "other" scores);
2. save the weights with `save_weights` (Ch. 3), keeping `best.weights.h5` for the best validation macro-F1 and `last.weights.h5` plus a small state file to **resume** after a disconnect.

`class_weight` (a standard `fit` argument, not covered in the book) makes the slightly rarer classes count a bit more in the loss. The values come from `03a`.

If the session drops, run 4.4 again and then this cell: it continues where it stopped.

In [ ]:
CW = {i: class_weights[c] for i, c in enumerate(CLASSES)}
STATE = CKPT_DIR / "state.json"
state = json.load(open(STATE)) if STATE.exists() else {"phase": "head", "epoch": -1, "best_f1": -1.0, "stale": 0, "history": []}

def evaluate_split(split):
    gen = flow(split, train=False)
    probs = model.predict(gen, verbose=0)
    y_true, y_pred = data[split][1], probs.argmax(1)
    loss = float(tf.keras.losses.sparse_categorical_crossentropy(y_true, probs).numpy().mean())
    f1 = f1_score(y_true, y_pred, average="macro", labels=range(len(CLASSES)), zero_division=0)
    return loss, float((y_true == y_pred).mean()), float(f1), y_true, y_pred

def load_w(path):
    # Restores the layer weights. Keras warns that the optimizer's state isn't restored; that's harmless
    # (Adam just restarts its running averages), so that one warning is hidden.
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
        model.load_weights(path)

def save_state():
    json.dump(state, open(STATE, "w"), indent=1)
    pd.DataFrame(state["history"]).to_csv(RESULTS_DIR / "results.csv", index=False)

def train_epochs(phase, n_epochs, early_stop):
    train_gen = flow("train", train=True)
    first = state["epoch"] + 1 if state["phase"] == phase else 0
    for epoch in range(first, n_epochs):
        t0 = time.time()
        h = model.fit(train_gen, epochs=1, class_weight=CW, verbose=0)
        va_loss, va_acc, va_f1, *_ = evaluate_split("val")
        row = dict(phase=phase, epoch=epoch, train_loss=h.history["loss"][0], train_acc=h.history["accuracy"][0],
                   val_loss=va_loss, val_acc=va_acc, val_macro_f1=va_f1, secs=time.time() - t0)
        state["history"].append(row)
        improved = va_f1 > state["best_f1"]
        if improved:
            state["best_f1"], state["stale"] = va_f1, 0
            model.save_weights(CKPT_DIR / "best.weights.h5")
        else:
            state["stale"] += 1
        model.save_weights(CKPT_DIR / "last.weights.h5")
        state.update(phase=phase, epoch=epoch); save_state()
        print(f"[{phase}] epoch {epoch:2d} | train loss {row['train_loss']:.3f} acc {row['train_acc']:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f} macro-F1 {va_f1:.3f}{' *' if improved else ''} | {row['secs']:.0f}s")
        if early_stop and state["stale"] >= HP["patience"]:
            print(f"Early stop: no improvement for {HP['patience']} epochs"); break

if state["history"] and (CKPT_DIR / "last.weights.h5").exists():          # resuming
    set_finetune(base_model, state["phase"] == "finetune")
    load_w(CKPT_DIR / "last.weights.h5")
    compile_model(model, HP["lr_finetune"] if state["phase"] == "finetune" else HP["lr_head"])
    print(f"Resumed: phase {state['phase']}, after epoch {state['epoch']} (best val macro-F1 {state['best_f1']:.3f})")

if state["phase"] == "head":
    train_epochs("head", HP["epochs_head"], early_stop=False)

## 4.7 Phase 2: fine-tuning (Ch. 3)

Now unfreeze part of the backbone, as in the textbook's *Fine-Tuning* section: early layers detect generic edges and textures and stay frozen; the last block learns features specific to our data. Only the **last block** is retrained (EfficientNetB0: block 7 and its top layers; VGG16: block 5; ResNet50: conv5), together with our head. BatchNormalization layers stay frozen throughout (see 4.4).

**One difference from the book:** the book keeps `Adam(0.001)` for fine-tuning. We use **0.0001**, because large updates can wreck the pretrained filters in the first few batches; ten times lower is the usual choice. Training stops early if validation macro-F1 hasn't improved for `patience` epochs.

In [ ]:
if state["phase"] == "head":
    set_finetune(base_model, True)
    compile_model(model, HP["lr_finetune"])        # must re-compile after changing `trainable`
    state.update(phase="finetune", epoch=-1, stale=0); save_state()
train_epochs("finetune", HP["epochs_finetune"], early_stop=True)
print(f"Best val macro-F1 overall: {state['best_f1']:.3f}")

## 4.8 Learning curves

The dashed line marks where fine-tuning starts. What to look for:
- **Both losses falling, then flattening:** healthy.
- **Train loss keeps falling while validation loss rises:** overfitting. Early stopping limits it; if it starts right away in fine-tuning, freeze more layers or lower `lr_finetune`.
- **A jump in validation loss right after the dashed line:** the fine-tuning learning rate is too high.

In [ ]:
h = pd.read_csv(RESULTS_DIR / "results.csv"); h["step"] = range(1, len(h) + 1)
switch = h.index[h.phase == "finetune"].min()
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
a.plot(h.step, h.train_loss, "o-", label="train"); a.plot(h.step, h.val_loss, "o-", label="validation"); a.set_ylabel("loss")
b.plot(h.step, h.train_acc, "o-", label="train accuracy"); b.plot(h.step, h.val_macro_f1, "o-", label="validation macro-F1")
b.set_ylim(0, 1)
for ax in (a, b):
    if not np.isnan(switch): ax.axvline(switch + 0.5, ls="--", c="gray")
    ax.set_xlabel("epoch (both phases)"); ax.legend(); ax.xaxis.get_major_locator().set_params(integer=True)
a.set_title("Loss"); b.set_title("Accuracy and macro-F1")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "curves.png", dpi=100); plt.show()

## 4.9 Test-set evaluation (run once)

Load the best weights (chosen on **validation**) and evaluate on the **test videos** once. These numbers go in the report. If you change something and retrain, that's a new run with a new `RUN_NAME`, not a second look at the test set to pick a winner.

We report accuracy (as in the book), the **confusion matrix**, and per-class precision, recall and F1. Finally the whole model is saved with `model.save` (Ch. 3, *Saving the Entire Model*; Keras 3 requires the `.keras` extension). **Step 5 loads this file** to reuse the fine-tuned backbone as its feature extractor.

In [ ]:
load_w(CKPT_DIR / "best.weights.h5")
te_loss, te_acc, te_f1, te_true, te_pred = evaluate_split("test")
report = classification_report(te_true, te_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3,
                               output_dict=True, zero_division=0)
best_row = max(state["history"], key=lambda r: r["val_macro_f1"])
print(f"best epoch: {best_row['phase']} {best_row['epoch']} (val macro-F1 {best_row['val_macro_f1']:.3f})")
print(f"TEST  macro-F1 {te_f1:.3f} | accuracy {te_acc:.3f}\n")
print(classification_report(te_true, te_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))
json.dump({"test_macro_f1": te_f1, "test_acc": te_acc, "best": best_row, "report": report},
          open(RESULTS_DIR / "test_report.json", "w"), indent=2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(te_true, te_pred, labels=range(len(CLASSES)))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=axes[0], colorbar=False, cmap="Blues"); axes[0].set_title("Counts")
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
ConfusionMatrixDisplay(cmn.round(2), display_labels=CLASSES).plot(ax=axes[1], colorbar=False, cmap="Blues"); axes[1].set_title("Row-normalised (recall)")
for ax in axes: ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.suptitle(f"{BACKBONE} single frame · test macro-F1 {te_f1:.3f}"); fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=100); plt.show()

model.save(CKPT_DIR / f"baseline_{BACKBONE}.keras")            # Step 5 loads this file
print("saved", CKPT_DIR / f"baseline_{BACKBONE}.keras")
summary_path = RESULTS_DIR.parent / "summary.csv"
row = {"model": f"{BACKBONE} single frame (baseline)", "input": "1 frame", "run": RUN, "test_macro_f1": round(te_f1, 3),
       "test_acc": round(te_acc, 3), **{f"{c}_f1": round(report[c]["f1-score"], 3) for c in CLASSES}}
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame(columns=list(row))
summary = pd.concat([summary[summary["run"] != RUN], pd.DataFrame([row])], ignore_index=True)
summary.to_csv(summary_path, index=False); summary

## 4.10 Look at the mistakes

Numbers tell you *how often* the model is wrong; pictures tell you *why*. Typical patterns: spike vs block (both in the air at the net; one frame can't show which way the arms move), set vs dig, and real actions called "other" when the player is small or blurred. **These are the cases the temporal models in Step 5 should fix**, so save this figure for the report.

In [ ]:
test_meta = data["test"][2]
wrong = np.where(te_pred != te_true)[0]
pick = np.random.RandomState(SEED).choice(wrong, size=min(16, len(wrong)), replace=False) if len(wrong) else []
if len(pick):
    fig, axes = plt.subplots(2, 8, figsize=(18, 6.6))
    for ax in axes.ravel(): ax.axis("off")
    for ax, i in zip(axes.ravel(), pick):
        r = test_meta.iloc[i]
        ax.imshow(data["test"][0][i]); ax.set_title(f"true {CLASSES[te_true[i]]}\npred {CLASSES[te_pred[i]]}\nv{r.vid} f{r.fid}", fontsize=9)
    fig.suptitle("Misclassified test examples (middle frame)"); fig.tight_layout()
    fig.savefig(RESULTS_DIR / "errors.png", dpi=90); plt.show()
print(f"{len(wrong)} of {len(te_true)} test clips misclassified")

## Done checklist (Step 4)

- [ ] 4.5: the sanity check printed OK
- [ ] Both phases finished (or fine-tuning stopped early); `curves.png` looks healthy
- [ ] Test macro-F1, accuracy and per-class F1 recorded in `summary.csv` (first row of the Step 6 table)
- [ ] `confusion_matrix.png` and `errors.png` saved for the report
- [ ] `baseline_efficientnetb0.keras` saved (Step 5 needs it)
- [ ] Optional: rerun with `BACKBONE = "vgg16"` for a "textbook backbone" row in the results table
- [ ] Commit this notebook and the `results/` CSV, JSON and PNG files; keep the `.weights.h5` and `.keras` files on Drive only

**Reading your result.** On a balanced 5-class test set, chance is 0.20 macro-F1. A single frame usually does well on dig and "other" (posture is distinctive) and worst on spike vs block. Whatever you get is the bar for Step 5 (`03b_temporal_models.ipynb`): the same backbone's features for all 16 frames, fed through the textbook's sequence models, `Conv1D` (Ch. 5) and `LSTM` / `GRU` (Ch. 6).